# Building a RAG System with Ollama, LangChain & ChromaDB

## Introduction

**Retrieval-Augmented Generation (RAG)** is a technique that combines a Large Language Model (LLM) with an external knowledge base to produce more accurate and grounded answers [1]. Instead of relying only on what the model learned during training, RAG first **retrieves** relevant documents and then **generates** a response based on that context [7].

This tutorial walks you through building a basic RAG system using three tools:

- **Ollama** — runs LLMs locally on your machine [6]
- **LangChain** — a framework for building LLM-powered applications [5]
- **ChromaDB** — an open-source vector database for storing and searching embeddings [4]

### System Architecture

The RAG pipeline has two main phases:

1. **Indexing**: Documents → Embeddings → Vector Store (ChromaDB)
2. **Querying**: User Question → Similarity Search → Retrieved Context → LLM → Answer

### References

- [1] Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020.
- [2] Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS 2017.
- [3] Mikolov, T. et al. (2013). *Efficient Estimation of Word Representations in Vector Space*. ICLR 2013.
- [4] Chroma. (2023). *ChromaDB: The AI-native open-source embedding database*. https://docs.trychroma.com
- [5] Chase, H. (2022). *LangChain: Building applications with LLMs through composability*. https://langchain.com
- [6] Ollama. (2023). *Run large language models locally*. https://ollama.com
- [7] Gao, Y. et al. (2024). *Retrieval-Augmented Generation for Large Language Models: A Survey*. arXiv:2312.10997.

## 1. Installation and Setup

First, we check that Ollama [6] is installed and running on the system.

In [26]:
# Check that Ollama is installed and running
import subprocess
import sys

try:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True, check=True)
    print(f"Ollama is installed: {result.stdout.strip()}")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("Ollama is NOT installed. Install it with:")
    print("   macOS: brew install ollama")
    print("   Linux: curl -fsSL https://ollama.ai/install.sh | sh")
    print("   Then run: ollama serve")

Ollama is installed: Warning: could not connect to a running Ollama instance


In [27]:
# Install all required dependencies
%pip install -q langchain langchain-community langchain-ollama langchain-core langchain-text-splitters
%pip install -q chromadb langchain-chroma
%pip install -q sentence-transformers

print("All dependencies installed successfully")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
All dependencies installed successfully


## 2. Download Required Models

We need two models from Ollama [6]:
- **Embedding model** (`nomic-embed-text`): Converts text into numerical vectors (embeddings) [3]
- **Chat model** (`llama3.2`): Generates natural language answers [6]

In [28]:
# Download the required Ollama models
import subprocess

def download_ollama_model(model_name):
    """Download an Ollama model if it is not already available."""
    try:
        # Check if the model already exists
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if model_name in result.stdout:
            print(f"Model {model_name} is already available")
            return True

        # Download the model
        print(f"Downloading model {model_name}...")
        subprocess.run(["ollama", "pull", model_name], check=True)
        print(f"Model {model_name} downloaded successfully")
        return True

    except subprocess.CalledProcessError as e:
        print(f"Error downloading {model_name}: {e}")
        return False
    except FileNotFoundError:
        print("Ollama is not in PATH. Make sure it is installed and running.")
        return False

# Embedding model for vector representations [3]
embedding_model = "nomic-embed-text"
# Chat model for generating answers [6]
chat_model = "llama3.2"

download_ollama_model(embedding_model)
download_ollama_model(chat_model)

Model nomic-embed-text is already available
Model llama3.2 is already available


True

## 3. Import Libraries

We import all the components from LangChain [5] that we will use throughout the tutorial.

In [29]:
# Import all required libraries
from langchain_ollama import OllamaEmbeddings, ChatOllama    # Ollama integration [6]
from langchain_chroma import Chroma                           # Vector store [4]
from langchain_core.documents import Document                 # Document wrapper [5]
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Text chunking [5]
from langchain_core.prompts import ChatPromptTemplate         # Prompt templates [5]
from langchain_core.runnables import RunnablePassthrough       # LCEL components [5]
from langchain_core.output_parsers import StrOutputParser      # Output parsing [5]
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("All libraries imported successfully")

All libraries imported successfully


## 4. Prepare Sample Documents

In a RAG system, the knowledge base is made of **documents** [1]. Here we create simple example documents. In a real project, these could come from PDFs, web pages, databases, or APIs.

In [30]:
# Create sample documents about Python programming
documents_data = [
    {
        "content": "Python is an interpreted, high-level, general-purpose programming language. It was created by Guido van Rossum and first released in 1991. Python emphasizes code readability with its notable use of significant whitespace.",
        "metadata": {"topic": "python_basics", "source": "intro"}
    },
    {
        "content": "Lists in Python are ordered, mutable data structures that can hold elements of different types. They are defined using square brackets [] and elements are separated by commas. Example: my_list = [1, 'hello', 3.14, True]",
        "metadata": {"topic": "python_data_structures", "source": "lists"}
    },
    {
        "content": "Functions in Python are defined using the 'def' keyword followed by the function name and parentheses. Functions can receive parameters and return values using the 'return' keyword. Example: def greet(name): return f'Hello, {name}'",
        "metadata": {"topic": "python_functions", "source": "functions"}
    },
    {
        "content": "Dictionaries in Python are data structures that store key-value pairs. They are mutable and since Python 3.7+ they maintain insertion order. They are defined using curly braces {}. Example: my_dict = {'name': 'Alice', 'age': 30}",
        "metadata": {"topic": "python_data_structures", "source": "dictionaries"}
    },
    {
        "content": "For loops in Python are used to iterate over sequences (lists, tuples, strings, etc.). Syntax: for item in sequence: # code. You can also use range() to iterate a specific number of times: for i in range(10): # code",
        "metadata": {"topic": "python_control_flow", "source": "loops"}
    }
]

# Convert raw data into LangChain Document objects [5]
documents = [
    Document(page_content=d["content"], metadata=d["metadata"])
    for d in documents_data
]

print(f"Created {len(documents)} sample documents")
print(f"\nExample: {documents[0].page_content[:80]}...")
print(f"Metadata: {documents[0].metadata}")

Created 5 sample documents

Example: Python is an interpreted, high-level, general-purpose programming language. It w...
Metadata: {'topic': 'python_basics', 'source': 'intro'}


## 5. Configure Embeddings with Ollama

**Embeddings** are numerical vector representations of text [3]. Words or sentences with similar meanings produce vectors that are close together in the vector space. This is what makes semantic search possible — we can find documents that are *related in meaning* to a query, not just documents that share exact keywords [2].

In [31]:
# Configure the Ollama embedding model [3]
embedding_model_name = "nomic-embed-text"
embeddings = OllamaEmbeddings(model=embedding_model_name)

# Test the embeddings with a sample text
test_text = "Python is a programming language"
test_embedding = embeddings.embed_query(test_text)

print(f"Embeddings working correctly")
print(f"Vector dimensions: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

Embeddings working correctly
Vector dimensions: 768
First 5 values: [0.014105377, 0.023323976, -0.10265142, -0.020855395, 0.07487801]


## 6. Set Up ChromaDB as a Vector Store

ChromaDB [4] is a vector database that stores embeddings and allows fast **similarity search**. When we receive a query, ChromaDB finds the stored documents whose embeddings are closest to the query embedding. This step is the core of the *retrieval* part of RAG [1].

In [32]:
# Create the vector store with ChromaDB [4]
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="python_docs",
)

print(f"Vector store created")
print(f"Documents stored: {vectorstore._collection.count()}")

# Create a retriever that returns the top-k most similar documents [1]
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}  # Return top 2 most relevant documents
)

print("Retriever configured successfully")

Vector store created
Documents stored: 20
Retriever configured successfully


## 7. Test the Retrieval System

Before building the full RAG pipeline, we verify that the semantic search is working correctly. Given a user query, the retriever should return the most relevant documents from our knowledge base [1].

In [33]:
# Test the retrieval system with different queries
test_queries = [
    "How do you create lists in Python?",
    "What are dictionaries?",
    "How do you define functions?",
    "What is Python?"
]

print("Testing the retrieval system:\n")

for query in test_queries:
    print(f"Query: {query}")

    # Retrieve relevant documents using similarity search [1]
    relevant_docs = retriever.invoke(query)

    print(f"Documents found: {len(relevant_docs)}")

    for i, doc in enumerate(relevant_docs, 1):
        print(f"  {i}. {doc.page_content[:100]}...")

    print("-" * 80)

Testing the retrieval system:

Query: How do you create lists in Python?
Documents found: 2
  1. Lists in Python are ordered, mutable data structures that can hold elements of different types. They...
  2. Lists in Python are ordered, mutable data structures that can hold elements of different types. They...
--------------------------------------------------------------------------------
Query: What are dictionaries?
Documents found: 2
  1. Dictionaries in Python are data structures that store key-value pairs. They are mutable and since Py...
  2. Dictionaries in Python are data structures that store key-value pairs. They are mutable and since Py...
--------------------------------------------------------------------------------
Query: How do you define functions?
Documents found: 2
  1. Functions in Python are defined using the 'def' keyword followed by the function name and parenthese...
  2. Functions in Python are defined using the 'def' keyword followed by the function name and pa

## 8. Configure the Chat Model

Now we set up the LLM that will **generate** answers. We use `llama3.2` through Ollama [6]. The model receives the retrieved context and the user's question, and produces a natural language response. This is the *generation* part of RAG [1].

In [34]:
# Configure the Ollama chat model [6]
chat_model_name = "llama3.2"
llm = ChatOllama(
    model=chat_model_name,
    temperature=0.1,        # Low temperature for more precise answers
    max_tokens=500          # Limit response length
)

# Test the chat model with a simple prompt
try:
    test_response = llm.invoke("Briefly explain what Python is")
    print("Chat model is working correctly")
    print(f"Test response:")
    print(f"{test_response.content}")
except Exception as e:
    print(f"Error with the chat model: {e}")
    print("Make sure Ollama is running and the model is downloaded")

Chat model is working correctly
Test response:
Python is a high-level, interpreted programming language that is widely used for various purposes such as:

1. Web development
2. Data analysis and science
3. Artificial intelligence and machine learning
4. Automation and scripting
5. Education and research

It's known for its simplicity, readability, and ease of use, making it a popular choice among beginners and experienced programmers alike. Python is often used in industries such as finance, healthcare, and scientific research due to its versatility and flexibility.

Some key features of Python include:

* Simple syntax
* Large standard library
* Extensive community support
* Cross-platform compatibility

Overall, Python is a powerful and versatile language that can be used for a wide range of applications.


## 9. Build the Full RAG Pipeline

We now connect all the components into a single chain using **LCEL** (LangChain Expression Language) [5]. The pipeline works as follows:

1. The user's **question** goes to the retriever, which finds relevant documents
2. The retrieved documents are formatted into a **context** string
3. A **prompt template** combines the context and question
4. The **LLM** generates an answer based only on the provided context [1]
5. The output parser extracts the final text

This approach, known as the "stuff" method, concatenates all retrieved documents into the prompt [7].

In [35]:
# Build the RAG chain using LCEL (LangChain Expression Language) [5]

# Define the prompt template that instructs the LLM to use only the context [1]
rag_template = """Answer the question based ONLY on the following context.
If the information is not in the context, say you do not have enough information.

Context:
{context}

Question: {question}

Answer:"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

# Helper function to format retrieved documents into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain with LCEL pipe operators [5]
# retriever | format_docs  ->  converts question into context text
# RunnablePassthrough()     ->  passes the question through unchanged
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG system created successfully with LCEL!")
print("Components:")
print("   - Embedding model: nomic-embed-text [3]")
print("   - Vector store: ChromaDB [4]")
print("   - Chat model: llama3.2 [6]")
print("   - Retriever: similarity search (k=2)")
print("\nUsage: rag_chain.invoke('your question here')")

RAG system created successfully with LCEL!
Components:
   - Embedding model: nomic-embed-text [3]
   - Vector store: ChromaDB [4]
   - Chat model: llama3.2 [6]
   - Retriever: similarity search (k=2)

Usage: rag_chain.invoke('your question here')


## 10. Test the RAG System

Let's test the full RAG pipeline with several questions. Notice how the model answers based on the retrieved context rather than its general training knowledge [1].

In [36]:
# Test the RAG system with different questions
test_questions = [
    "How do you create lists in Python?",
    "What is the difference between lists and dictionaries?",
    "How do you define functions in Python?",
    "What are for loops used for?",
    "Who created Python and when?"
]

print("RAG SYSTEM TESTS")
print("=" * 80)

for question in test_questions:
    print(f"\nQuestion: {question}")
    print("-" * 60)

    answer = rag_chain.invoke(question)

    print(f"Answer: {answer}")
    print("=" * 80)

RAG SYSTEM TESTS

Question: How do you create lists in Python?
------------------------------------------------------------
Answer: You create lists in Python by defining them with square brackets [] and separating the elements with commas. For example:

my_list = [1, 'hello', 3.14, True]

Question: What is the difference between lists and dictionaries?
------------------------------------------------------------
Answer: I don't have enough information to answer this question accurately. The provided context only describes lists in Python, but it does not mention dictionaries.

Question: How do you define functions in Python?
------------------------------------------------------------
Answer: You define functions in Python using the 'def' keyword followed by the function name and parentheses.

Question: What are for loops used for?
------------------------------------------------------------
Answer: For loops are used to iterate over sequences (lists, tuples, strings, etc.) and to ite

## 11. Interactive Usage

You can use `rag_chain.invoke()` to ask your own questions. Change the question below and run the cell.

In [38]:
# Ask your own question to the RAG system
# Change the question and run this cell

question = "How can I use Python to create spaceships?"

print(f"Question: {question}")
print("-" * 60)
answer = rag_chain.invoke(question)
print(f"Answer: {answer}")

Question: How can I use Python to create spaceships?
------------------------------------------------------------
Answer: I don't have enough information to provide a specific answer on how to use Python to create spaceships, as the context only provides general information about the language and does not mention any specific application or library for creating spaceships.


## 12. Optional Cleanup

If you want to delete the vector store created during this tutorial, uncomment and run the cell below.

In [39]:
# Optional cleanup - uncomment to delete the vector store
# vectorstore.delete_collection()
# print("Vector store deleted successfully")

print("Tutorial complete. The vector store remains in memory.")

Tutorial complete. The vector store remains in memory.


## Conclusions and Next Steps

You have built a working RAG system from scratch using open-source tools running locally on your machine.

### What We Learned

1. **Ollama** [6] — How to run LLMs locally without cloud APIs
2. **Embeddings** [3] — How to convert text into numerical vectors for semantic search
3. **ChromaDB** [4] — How to store and search vectors efficiently
4. **LangChain** [5] — How to connect components into an LLM application using LCEL
5. **RAG** [1] — How to combine retrieval and generation for accurate, grounded answers

### System Architecture Summary

```
Question --> Embeddings --> ChromaDB --> Relevant Documents --> LLM --> Answer
```

### Possible Improvements

- **More documents**: Add diverse content to the knowledge base
- **Smart chunking**: Use better text splitting strategies for long documents [7]
- **Metadata filters**: Filter retrieved documents by metadata fields
- **Evaluation**: Measure answer quality with metrics like faithfulness and relevance [7]
- **Web interface**: Build a UI with Streamlit or Gradio
- **Persistent storage**: Save the vector store to disk for reuse across sessions

### References

- [1] Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. arXiv:2005.11401
- [2] Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS 2017. arXiv:1706.03762
- [3] Mikolov, T. et al. (2013). *Efficient Estimation of Word Representations in Vector Space*. ICLR 2013. arXiv:1301.3781
- [4] Chroma. (2023). *ChromaDB: The AI-native open-source embedding database*. https://docs.trychroma.com
- [5] Chase, H. (2022). *LangChain: Building applications with LLMs through composability*. https://langchain.com
- [6] Ollama. (2023). *Run large language models locally*. https://ollama.com
- [7] Gao, Y. et al. (2024). *Retrieval-Augmented Generation for Large Language Models: A Survey*. arXiv:2312.10997